In [1]:
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

In [2]:
import sys
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

sys.path.append("src")
from entropy_pruning import (
    AttentionForecaster,
    UNILoRAClassifier,
    build_attention_cache,
    build_loaders,
    set_seed,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

/home/vcivale/miniconda3/envs/entropy_pruning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


## Configuration

Cambia `DATASET_NAME` per switchare dataset (`NCT-CRC-HE` o `BREAKHIS`).

`COMBINATIONS` definisce la griglia `(src, tgt)` da addestrare. I checkpoint già esistenti vengono skippati automaticamente.

In [3]:
DATASET_NAME = "NCT-CRC-HE"   # ← cambia qui: "NCT-CRC-HE" oppure "BREAKHIS"

# griglia src × tgt da addestrare
SRC_LAYERS = [0, 1, 2, 3, 4, 5, 6]
TGT_LAYERS = [20, 21, 22, 23]
COMBINATIONS = [(s, t) for s in SRC_LAYERS for t in TGT_LAYERS]

CFG = dict(
    batch_size   = 512,
    num_workers  = 4,
    seed         = 42,
    epochs       = 30,
    lr           = 1e-4,
    weight_decay = 0.05,
)

DATA_ROOTS = {
    "NCT-CRC-HE": "/raid/DATASETS/NCT-CRC-HE",
    "BREAKHIS":   "/raid/DATASETS/BREAKHIS",
}

data_dir        = DATA_ROOTS[DATASET_NAME]
classifier_ckpt = Path(f"/raid/DATASETS/checkpoints-Attention-Pruning/{DATASET_NAME}/uni_finetuned/best_model.pt")
cache_path      = Path(f"{data_dir}/data_cache/{DATASET_NAME}_src_tgt_ablation.h5")
forecaster_dir  = Path(f"/raid/DATASETS/checkpoints-Attention-Pruning/{DATASET_NAME}/forecaster_ablation")
forecaster_dir.mkdir(parents=True, exist_ok=True)

set_seed(CFG["seed"])

print(f"Dataset      : {DATASET_NAME}")
print(f"Classifier   : {classifier_ckpt}")
print(f"Cache        : {cache_path}")
print(f"Forecasters  : {forecaster_dir}")
print(f"Combinations : {len(COMBINATIONS)}  ({SRC_LAYERS} × {TGT_LAYERS})")

Dataset      : NCT-CRC-HE
Classifier   : /raid/DATASETS/checkpoints-Attention-Pruning/NCT-CRC-HE/uni_finetuned/best_model.pt
Cache        : /raid/DATASETS/NCT-CRC-HE/data_cache/NCT-CRC-HE_src_tgt_ablation.h5
Forecasters  : /raid/DATASETS/checkpoints-Attention-Pruning/NCT-CRC-HE/forecaster_ablation
Combinations : 28  ([0, 1, 2, 3, 4, 5, 6] × [20, 21, 22, 23])


## Step 1 — Build multi-layer cache

Estrae embeddings da tutti i layer source e attention maps da tutti i layer target.
Se la cache esiste già viene skippato.

In [4]:
cache_path.parent.mkdir(parents=True, exist_ok=True)

# Check if existing cache has all required layers
needs_rebuild = False
if cache_path.exists():
    with h5py.File(cache_path, "r") as f:
        existing_keys = set(f[list(f.keys())[0]].keys())
    needed_src = {f"emb_layer{s}" for s in SRC_LAYERS}
    needed_tgt = {f"attn_layer{t}" for t in TGT_LAYERS}
    missing = (needed_src | needed_tgt) - existing_keys
    if missing:
        print(f"Cache exists but missing keys: {sorted(missing)}")
        print("Rebuilding cache with all required layers...")
        needs_rebuild = True
        cache_path.unlink()  # remove incomplete cache
    else:
        print(f"Cache OK: {cache_path}")
        with h5py.File(cache_path, "r") as f:
            for split in f:
                keys = list(f[split].keys())
                print(f"  [{split}] {len(f[split]['labels']):,} samples — keys: {keys}")

if not cache_path.exists() or needs_rebuild:
    print("Costruzione cache multi-layer...")
    loaders = build_loaders(
        data_dir    = data_dir,
        img_size    = 224,
        batch_size  = CFG["batch_size"],
        num_workers = CFG["num_workers"],
        drop_last_train = False,
    )
    model = UNILoRAClassifier(loaders.n_classes).to(device)
    model.load_state_dict(torch.load(classifier_ckpt, map_location=device), strict=False)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)

    build_attention_cache(
        model         = model,
        loaders       = {
            "train": loaders.train_loader,
            "val":   loaders.val_loader,
            "test":  loaders.test_loader,
        },
        device        = device,
        source_layers = SRC_LAYERS,
        target_layers = TGT_LAYERS,
        save_path     = cache_path,
    )
    del model
    torch.cuda.empty_cache()
    print(f"Cache salvata → {cache_path}")

Cache OK: /raid/DATASETS/NCT-CRC-HE/data_cache/NCT-CRC-HE_src_tgt_ablation.h5
  [test] 7,180 samples — keys: ['attn_layer20', 'attn_layer21', 'attn_layer22', 'attn_layer23', 'emb_layer0', 'emb_layer1', 'emb_layer2', 'emb_layer3', 'emb_layer4', 'emb_layer5', 'emb_layer6', 'labels']
  [train] 90,000 samples — keys: ['attn_layer20', 'attn_layer21', 'attn_layer22', 'attn_layer23', 'emb_layer0', 'emb_layer1', 'emb_layer2', 'emb_layer3', 'emb_layer4', 'emb_layer5', 'emb_layer6', 'labels']
  [val] 10,000 samples — keys: ['attn_layer20', 'attn_layer21', 'attn_layer22', 'attn_layer23', 'emb_layer0', 'emb_layer1', 'emb_layer2', 'emb_layer3', 'emb_layer4', 'emb_layer5', 'emb_layer6', 'labels']


## Step 2 — Train missing forecasters

Per ogni combinazione `(src, tgt)` nella griglia:
- se il checkpoint esiste già → skip
- altrimenti → carica embeddings dalla cache, addestra, salva

In [5]:
def rankdata_2d(x: np.ndarray) -> np.ndarray:
    idx   = np.argsort(x, axis=1)
    ranks = np.empty_like(idx, dtype=float)
    np.put_along_axis(ranks, idx, np.arange(1, x.shape[1] + 1, dtype=float)[None], axis=1)
    return ranks

def spearman_matrix(pred_np: np.ndarray, tgt_np: np.ndarray) -> np.ndarray:
    rp, rt = rankdata_2d(pred_np), rankdata_2d(tgt_np)
    n      = pred_np.shape[1]
    return 1 - 6 * ((rp - rt) ** 2).sum(axis=1) / (n * (n ** 2 - 1))


def load_split(h5_path, split, src_layer, tgt_layer):
    with h5py.File(h5_path, "r") as f:
        grp    = f[split]
        emb    = torch.from_numpy(grp[f"emb_layer{src_layer}"][:]).float()
        target = torch.from_numpy(grp[f"attn_layer{tgt_layer}"][:]).float()
        labels = torch.from_numpy(grp["labels"][:]).long()
    return TensorDataset(emb, target, labels)


def make_loaders(h5_path, src_layer, tgt_layer, batch_size, num_workers):
    kw = dict(batch_size=batch_size, num_workers=num_workers,
              pin_memory=True, persistent_workers=True)
    train_ds = load_split(h5_path, "train", src_layer, tgt_layer)
    val_ds   = load_split(h5_path, "val",   src_layer, tgt_layer)
    test_ds  = load_split(h5_path, "test",  src_layer, tgt_layer)
    return (
        DataLoader(train_ds, shuffle=True,  **kw),
        DataLoader(val_ds,   shuffle=False, **kw),
        DataLoader(test_ds,  shuffle=False, **kw),
    )


def train_forecaster_fast(model, save_path, train_loader, val_loader, test_loader, cfg, device):
    model = model.to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])

    best_val_kl  = float("inf")
    best_val_rho = -1.0

    for epoch in tqdm(range(cfg["epochs"]), desc="epochs", leave=False):
        # train
        model.train()
        for emb, target, _ in tqdm(train_loader, leave=False):
            emb, target = emb.to(device, non_blocking=True), target.to(device, non_blocking=True)
            pred = model(emb)
            loss = F.kl_div((pred + 1e-8).log(), target + 1e-8, reduction="batchmean") \
                 + 0.1 * F.mse_loss(pred, target)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        # val
        model.eval()
        val_kl_acc = torch.zeros(1, device=device)
        all_pred, all_tgt = [], []
        with torch.no_grad():
            for emb, target, _ in val_loader:
                emb, target = emb.to(device, non_blocking=True), target.to(device, non_blocking=True)
                pred = model(emb)
                val_kl_acc += F.kl_div((pred + 1e-8).log(), target + 1e-8, reduction="batchmean")
                all_pred.append(pred)
                all_tgt.append(target)

        val_kl  = (val_kl_acc / len(val_loader)).item()
        val_rho = float(np.nanmean(spearman_matrix(
            torch.cat(all_pred).cpu().numpy(),
            torch.cat(all_tgt).cpu().numpy(),
        )))

        if val_kl < best_val_kl:
            best_val_kl  = val_kl
            best_val_rho = val_rho
            torch.save(model.state_dict(), save_path)

        sched.step()

    # test evaluation with best checkpoint
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval()
    test_rhos = []
    with torch.no_grad():
        for emb, target, _ in test_loader:
            pred = model(emb.to(device, non_blocking=True)).cpu().numpy()
            test_rhos.extend(spearman_matrix(pred, target.numpy()).tolist())

    return {
        "best_val_kl":  best_val_kl,
        "best_val_rho": best_val_rho,
        "test_rho":     float(np.nanmean(test_rhos)),
    }

print("Utility functions ready.")

Utility functions ready.


In [6]:
all_results = []
n_total     = len(COMBINATIONS)

for i, (src, tgt) in enumerate(COMBINATIONS):
    ckpt_path = forecaster_dir / f"forecaster_src{src:02d}_tgt{tgt:02d}.pt"

    if ckpt_path.exists():
        print(f"[{i+1:2d}/{n_total}] src={src:02d} tgt={tgt:02d}  → SKIP (già esiste)")
        # valuta comunque per raccogliere le metriche
        _, _, test_loader = make_loaders(cache_path, src, tgt,
                                         CFG["batch_size"], CFG["num_workers"])
        model = AttentionForecaster(embed_dim=1024, hidden=256, n_heads=4, n_layers=2, dropout=0.1)
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        model.to(device).eval()
        test_rhos = []
        with torch.no_grad():
            for emb, target, _ in test_loader:
                pred = model(emb.to(device, non_blocking=True)).cpu().numpy()
                test_rhos.extend(spearman_matrix(pred, target.numpy()).tolist())
        result = {"best_val_kl": None, "best_val_rho": None,
                  "test_rho": float(np.nanmean(test_rhos))}
        del model
        torch.cuda.empty_cache()
    else:
        print(f"[{i+1:2d}/{n_total}] src={src:02d} tgt={tgt:02d}  → TRAINING...")
        train_loader, val_loader, test_loader = make_loaders(
            cache_path, src, tgt, CFG["batch_size"], CFG["num_workers"]
        )
        model  = AttentionForecaster(embed_dim=1024, hidden=256, n_heads=4, n_layers=2, dropout=0.1)
        result = train_forecaster_fast(
            model, ckpt_path, train_loader, val_loader, test_loader, CFG, device
        )
        del model
        torch.cuda.empty_cache()

    print(f"          test_rho={result['test_rho']:.4f}  "
          + (f"val_kl={result['best_val_kl']:.4f}" if result['best_val_kl'] is not None else ""))

    all_results.append({"src": src, "tgt": tgt, **result})

print("\nDone.")

[ 1/28] src=00 tgt=20  → SKIP (già esiste)
          test_rho=0.6788  
[ 2/28] src=00 tgt=21  → SKIP (già esiste)
          test_rho=0.6825  
[ 3/28] src=00 tgt=22  → SKIP (già esiste)
          test_rho=0.6666  
[ 4/28] src=00 tgt=23  → SKIP (già esiste)
          test_rho=0.6768  
[ 5/28] src=01 tgt=20  → SKIP (già esiste)
          test_rho=0.6705  
[ 6/28] src=01 tgt=21  → SKIP (già esiste)
          test_rho=0.6729  
[ 7/28] src=01 tgt=22  → SKIP (già esiste)
          test_rho=0.6700  
[ 8/28] src=01 tgt=23  → SKIP (già esiste)
          test_rho=0.6744  
[ 9/28] src=02 tgt=20  → SKIP (già esiste)
          test_rho=0.7484  
[10/28] src=02 tgt=21  → SKIP (già esiste)
          test_rho=0.7597  
[11/28] src=02 tgt=22  → SKIP (già esiste)
          test_rho=0.7505  
[12/28] src=02 tgt=23  → SKIP (già esiste)
          test_rho=0.7579  
[13/28] src=03 tgt=20  → SKIP (già esiste)
          test_rho=0.7146  
[14/28] src=03 tgt=21  → SKIP (già esiste)
          test_rho=0.7217  
[15/28

In [7]:
import pandas as pd

df = pd.DataFrame(all_results)
Path("results").mkdir(exist_ok=True)
out_csv = f"results/src_tgt_ablation_{DATASET_NAME}.csv"
df.to_csv(out_csv, index=False)
print(f"Saved → {out_csv}")
df.pivot(index="src", columns="tgt", values="test_rho").round(4)

Saved → results/src_tgt_ablation_NCT-CRC-HE.csv


tgt,20,21,22,23
src,,,,
0,0.6788,0.6825,0.6666,0.6768
1,0.6705,0.6729,0.6700,0.6744
2,0.7484,0.7597,0.7505,0.7579
3,0.7146,0.7217,0.7165,0.7256
4,0.7182,0.7209,0.7201,0.7235
5,0.7339,0.7433,0.7440,0.7442
6,0.7864,0.7883,0.7859,0.7857
